# Random Forest - Cross Validation (SEPSIS)

Notebook para executar Random Forest com StratifiedKFold sobre o dataset SEPSIS.
- Selecione qual dataset normalizado usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `rf_params`.
- Outputs: `model_reports/random_forest/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/SEPSIS/common/models/random_forest/v1/rf_model_cf{N_SPLITS}.pkl`.

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Escolha um dos 3 datasets normalizados gerados pelo EDA: 'Standard', 'Robust', 'MinMax'
DATASET_NAME = 'exams.csv'  # alterar para exams.csv ou exams_robust_scaled.csv ou exams_minmax_scaled.csv
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Exemplo: TARGET_COLUMN = 'diagnosis' ou TARGET_COLUMN = 'y'
TARGET_COLUMN = 'septic' # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/random_forest/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/random_forest/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# Random Forest params (variável conforme solicitado)
rf_params = {
    "n_estimators": 650,
    "max_depth": 16,
    "min_samples_split": 8,
    "min_samples_leaf": 15,
    "max_features": 0.3,
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "criterion": "entropy",
    'n_jobs': 8
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
#EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id']
#EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'Chloride (serum)' , 'Creatinine (serum)', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other','gender_M']
EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other']


# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10%% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TARGET_COLUMN = 'septic'  # ajustar se necessário
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42


In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)

def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'septic', 'target', 'label', 'class']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid', 'septic', 'stay_id'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
if not np.issubdtype(df['y'].dtype, np.number):
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= set(['0', '1']):
        mapping = {v: (1 if str(v).lower() == '1' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
    else:
        # factorize para inteiros 0..k-1
        df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\exams_raw.csv
Dataset shape: (94446, 33)


In [ ]:
# ============================ 8/1/1 NESTED-STYLE COM OPTUNA (RF) ============================
# Requisitos: optuna, scikit-learn
# Espera: df com coluna alvo 'y' e EXCLUDE_COLUMNS (IDs etc). Todas as features já numéricas.

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.ensemble import RandomForestClassifier

# -------------------- CONFIG --------------------
RANDOM_STATE     = 42
N_TRIALS         = 5          # ajuste conforme recurso/tempo
N_SPLITS_OUTER   = 10
N_SPLITS_INNER   = 8           # CV interna nos 8 folds de treino
THRESHOLD        = 0.5         # limiar de classificação
EPS              = 1e-12       # estabilidade para logs na cross-entropy

# ---------- separa X,y sem vazamento ----------
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
id_cols = [c for c in df.columns if c.lower() in exclude_lower]
id_cols = [c for c in id_cols if c != 'y']
FEATURES = [c for c in df.columns if c not in id_cols + ['y']]
X_raw = df[FEATURES].copy()
y = df['y'].copy()

# ---------- define outer folds (10) e fixa quem é teste e validação ----------
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)

# ---------------------- Correção 
#fold_indices = list(outer_kf.split(X_raw, y))

# Escolha determinística: último fold como TESTE, penúltimo como VALID, os demais 8 = TREINO
#(train_val_indices, test_indices)   = fold_indices[-1]
#(train_inner_indices, val_indices)  = fold_indices[-2]

# Conjuntos efetivos
#train8_idx = train_inner_indices   # 8 folds para busca
#val1_idx   = val_indices           # 1 fold de validação intermediária
#test1_idx  = test_indices          # 1 fold final de teste


# ---------- define outer folds (10) e fixa quem é treino(8), validação(1) e teste(1) ----------
outer_kf = StratifiedKFold(
    n_splits=N_SPLITS_OUTER,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Cada elemento de folds é um "bloco" estratificado de índices (test_idx de cada fold)
folds = []
for _, test_idx in outer_kf.split(X_raw, y):
    folds.append(test_idx)

# Agora definimos explicitamente o esquema 8/1/1:
# - 8 primeiros folds  -> treino (busca de hiperparâmetros)
# - 9º fold            -> validação intermediária
# - 10º fold           -> teste final (nunca visto antes)
import numpy as np

train8_idx = np.concatenate(folds[:8])  # ~80% do total (569 → ~455)
val1_idx   = folds[8]                   # ~10%
test1_idx  = folds[9]                   # ~10%

# Conjuntos efetivos (sem vazamento)
X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

# --------------- Correção

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

# ---------- OPTUNA na CV interna (8 folds) ----------
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def build_model(trial):
    # Espaço de busca
    n_estimators      = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth_choice  = trial.suggest_categorical("max_depth_choice", ["None", "int"])
    max_depth         = None if max_depth_choice == "None" else trial.suggest_int("max_depth", 3, 50, step=1)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20, step=1)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 20, step=1)
    bootstrap         = trial.suggest_categorical("bootstrap", [True, False])
    class_weight      = trial.suggest_categorical("class_weight", [None, "balanced"])
    max_features_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "float"])
    if max_features_mode == "float":
        max_features = trial.suggest_float("max_features_float", 0.2, 1.0)
    else:
        max_features = max_features_mode
    max_samples = None
    if bootstrap:
        max_samples = trial.suggest_float("max_samples", 0.5, 1.0)

    rf_params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    if bootstrap and (max_samples is not None):
        rf_params["max_samples"] = max_samples

    clf = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**rf_params))
    ])
    return clf

def objective(trial):
    clf = build_model(trial)
    aucs = []
    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, proba)
        aucs.append(auc)
        trial.report(float(auc), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

study = optuna.create_study(
    study_name="rf_optuna_8fold_search_numeric_ready",
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=2),
)
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

# ---------- reconstrói best_params ----------
best = study.best_trial
bp = dict(best.params)
if bp.get("max_depth_choice") == "None":
    bp["max_depth"] = None
bp["max_features"] = bp.get("max_features_float", bp.get("max_features_mode"))
for k in ["max_depth_choice", "max_features_mode", "max_features_float"]:
    bp.pop(k, None)
if not bp.get("bootstrap", True):
    bp.pop("max_samples", None)
bp["n_jobs"] = 8
bp["random_state"] = RANDOM_STATE
rf_best_params = bp

print("Melhor AUC (CV interna 8 folds):", best.value)
print("Melhores hiperparâmetros:", rf_best_params)

# ===================== FUNÇÕES DE MÉTRICAS (inclui Cross-Entropy por classe) =====================

def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    """
    y_true: [0/1]; p_pos: probabilidade prevista da classe 1.
    Retorna: CE0 (média de -log(1-p) em y=0) e CE1 (média de -log(p) em y=1).
    """
    p_pos = np.clip(p_pos, eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = ~mask0
    # pode não haver instâncias de alguma classe em um fold – proteja
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def all_metrics_table(y_true, proba, threshold=0.5, eps=1e-12):
    """
    Retorna um dict com TODAS as métricas solicitadas:
    - Acurácia, Cross-Entropy C0/C1, Proporção C0/C1, F1, Balanced Accuracy,
      Precision, Recall, TP/FP/TN/FN (abs) e TP/FP/TN/FN (% do total).
    """
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= threshold).astype(int)

    # Contagens
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    # Proporções por classe (no y_true)
    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    # Cross-Entropy por classe
    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps=eps)

    # Métricas padrão
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)

    # Balanced Accuracy = (recall_0 + recall_1)/2
    # recall_1 (sensibilidade) = TP/(TP+FN) ; recall_0 (especificidade) = TN/(TN+FP)
    recall_1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    recall_0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    bal_acc  = np.nanmean([recall_0, recall_1])

    # Percentuais de confusão
    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "Acurácia": acc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bal_acc,
        "Precision": prec,
        "Recall": rec,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
    }

# ===================== (1) Reavalia a CV interna (8 folds) COM os melhores params =====================
inner_fold_rows = []
inner_clf = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])

for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]
    inner_clf.fit(X_tr, y_tr)
    proba = inner_clf.predict_proba(X_va)[:, 1]
    metrics = all_metrics_table(y_va, proba, threshold=THRESHOLD, eps=EPS)
    row = {"etapa": "inner_cv", "fold": fold_id, **metrics}
    inner_fold_rows.append(row)

df_inner = pd.DataFrame(inner_fold_rows)
df_inner.to_csv("cv_811_inner_folds.csv", index=False)

# ===================== (2) Validação intermediária (fold 9) =====================
clf_val = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])
clf_val.fit(X_train8, y_train8)  # treina nos 8 folds
proba_val = clf_val.predict_proba(X_val1)[:, 1]
val_metrics = all_metrics_table(y_val1, proba_val, threshold=THRESHOLD, eps=EPS)
df_val = pd.DataFrame([{"etapa": "validacao", "fold": 9, **val_metrics}])
df_val.to_csv("cv_811_validation.csv", index=False)

# ===================== (3) Treino final (8+1) e teste (fold 10) =====================
X_train9 = pd.concat([X_train8, X_val1])
y_train9 = pd.concat([y_train8, y_val1])

clf_final = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])
clf_final.fit(X_train9, y_train9)
proba_test = clf_final.predict_proba(X_test1)[:, 1]
test_metrics = all_metrics_table(y_test1, proba_test, threshold=THRESHOLD, eps=EPS)
df_test = pd.DataFrame([{"etapa": "teste", "fold": 10, **test_metrics}])
df_test.to_csv("cv_811_test.csv", index=False)

# ===================== (4) Consolida tudo (formato longo) =====================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)
df_all.to_csv("cv_811_results_long.csv", index=False)

# ===================== Impressão resumida =====================
pd.options.display.float_format = lambda x: f"{x:.6f}"

print("\n===== Desempenho por etapa/fold (long) =====")
print(df_all.to_string(index=False))

print("\n===== Resumo por etapa (média ± desvio) =====")
summary = (df_all
           .groupby("etapa")[[
               "Acurácia","Cross-Entropy C0","Proporção C0","Cross-Entropy C1","Proporção C1",
               "F1 Score","Balanced Accuracy","Precision","Recall",
               "TP","FP","TN","FN","TP(%)","FP(%)","TN(%)","FN(%)"
           ]]
           .agg(['mean','std'])
          )
print(summary)


In [6]:
# ============================ 8/1/1 NESTED-STYLE COM OPTUNA (RF) ============================
# Requisitos: optuna, scikit-learn
# Espera: df com coluna alvo 'y' e EXCLUDE_COLUMNS (IDs etc). Todas as features já numéricas.

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
import datetime
from pathlib import Path

# -------------------- CONFIG --------------------
RANDOM_STATE     = 42
N_TRIALS         = 1          # ajuste conforme recurso/tempo
N_SPLITS_OUTER   = 10
N_SPLITS_INNER   = 8           # CV interna nos 8 folds de treino
THRESHOLD        = 0.5         # limiar de classificação (será sobrescrito)
EPS              = 1e-12       # estabilidade para logs na cross-entropy

# --- Configuração de Saída ---
if 'CSV_OUT' not in globals():
    CSV_OUT = Path('./cv_out')
CSV_OUT.mkdir(parents=True, exist_ok=True)


# ---------- separa X,y sem vazamento ----------
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS} if 'EXCLUDE_COLUMNS' in globals() else set()
id_cols = [c for c in df.columns if c.lower() in exclude_lower]
id_cols = [c for c in id_cols if c != 'y']
FEATURES = [c for c in df.columns if c not in id_cols + ['y']]
X_raw = df[FEATURES].copy()
y = df['y'].copy()

# ---------- define outer folds (10) e fixa quem é treino(8), validação(1) e teste(1) ----------
outer_kf = StratifiedKFold(
    n_splits=N_SPLITS_OUTER,
    shuffle=True,
    random_state=RANDOM_STATE
)

folds = []
for _, test_idx in outer_kf.split(X_raw, y):
    folds.append(test_idx)

train8_idx = np.concatenate(folds[:8])
val1_idx   = folds[8]
test1_idx  = folds[9]

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

# ---------- OPTUNA na CV interna (8 folds) ----------
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def build_model(trial):
    # Espaço de busca de hiperparâmetros (mantido do código original)
    n_estimators      = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth_choice  = trial.suggest_categorical("max_depth_choice", ["None", "int"])
    max_depth         = None if max_depth_choice == "None" else trial.suggest_int("max_depth", 3, 50, step=1)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20, step=1)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 20, step=1)
    bootstrap         = trial.suggest_categorical("bootstrap", [True, False])
    class_weight      = trial.suggest_categorical("class_weight", [None, "balanced"])
    max_features_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "float"])
    if max_features_mode == "float":
        max_features = trial.suggest_float("max_features_float", 0.2, 1.0)
    else:
        max_features = max_features_mode
    max_samples = None
    if bootstrap:
        max_samples = trial.suggest_float("max_samples", 0.5, 1.0)

    rf_params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    if bootstrap and (max_samples is not None):
        rf_params["max_samples"] = max_samples

    # O pipeline evita vazamento de dados
    clf = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**rf_params))
    ])
    return clf

def objective(trial):
    clf = build_model(trial)
    aucs = []
    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, proba)
        aucs.append(auc)
        trial.report(float(auc), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

study = optuna.create_study(
    study_name="rf_optuna_8fold_search_numeric_ready",
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=2),
)
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

# ---------- reconstrói best_params ----------
best = study.best_trial
bp = dict(best.params)
if bp.get("max_depth_choice") == "None":
    bp["max_depth"] = None
bp["max_features"] = bp.get("max_features_float", bp.get("max_features_mode"))
for k in ["max_depth_choice", "max_features_mode", "max_features_float"]:
    bp.pop(k, None)
if not bp.get("bootstrap", True):
    bp.pop("max_samples", None)
bp["n_jobs"] = 8
bp["random_state"] = RANDOM_STATE
rf_best_params = bp

print("Melhor AUC (CV interna 8 folds):", best.value)
print("Melhores hiperparâmetros:", rf_best_params)

# ===================== FUNÇÕES DE MÉTRICAS (inclui Cross-Entropy por classe) =====================

def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    p_pos = np.clip(p_pos, eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = ~mask0
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def all_metrics_table(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan
    
    # Métricas
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    auc_score = roc_auc_score(y_true, proba)
    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps)

    # % do total (conforme imagem anexa)
    tp_p = TP / total * 100 if total > 0 else np.nan
    tn_p = TN / total * 100 if total > 0 else np.nan
    fp_p = FP / total * 100 if total > 0 else np.nan
    fn_p = FN / total * 100 if total > 0 else np.nan

    return {
        'ACC': acc, 'F1': f1, 'BAC': bal_acc, 'PRE': prec, 'REC': rec, 'AUC': auc_score,
        'CE-C0': ce0, 'CE-C1': ce1, 'TP%': tp_p, 'TN%': tn_p, 'FP%': fp_p, 'FN%': fn_p,
        'TP_abs': TP, 'TN_abs': TN, 'FP_abs': FP, 'FN_abs': FN, 'N_total': total,
        'Threshold': threshold
    }

# ===================== TREINO FINAL, THRESHOLD OTMIZADO E AVALIAÇÃO =====================

# 1. Treinar o modelo final com os melhores hiperparâmetros em X_train8 completo
final_clf = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])
final_clf.fit(X_train8, y_train8)

# 2. Encontrar o threshold ideal para F1 na base de VALIDAÇÃO (Fold 9)
probas_val = final_clf.predict_proba(X_val1)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val1, probas_val)

# Calcular F1 para cada threshold e encontrar o máximo
f1_scores = 2 * recalls * precisions / (recalls + precisions + EPS)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"\nMelhor Threshold para F1 (na validação): {best_threshold:.4f}")

# 3. Avaliar o modelo final usando o best_threshold na base de TESTE (Fold 10)
probas_test = final_clf.predict_proba(X_test1)[:, 1]
test_metrics = all_metrics_table(y_test1, probas_test, threshold=best_threshold)

# 4. Imprimir resultados na tela
print("\n" + "="*40)
print(" RESULTADO FINAL NA BASE DE TESTE (Fold 10) ".center(40, '='))
print("="*40)
for metric, value in test_metrics.items():
    if isinstance(value, float):
        print(f" - {metric}: {value:.4f}")
    else:
        print(f" - {metric}: {value}")
print("="*40)

# 5. Salvar resultados em arquivo CSV (incluindo o timestamp)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f'final_test_metrics_{timestamp}.csv'
output_path = CSV_OUT / output_filename

try:
    pd.DataFrame([test_metrics]).to_csv(output_path, index=False)
    print(f"\n✅ Arquivo salvo com sucesso em: {output_path.resolve()}")
except PermissionError:
    print(f"\n❌ ERRO DE PERMISSÃO: Não foi possível salvar o arquivo. Feche programas como o Excel que possam estar acessando a pasta '{CSV_OUT.resolve()}' e tente novamente.")



[I 2026-01-26 05:17:43,814] A new study created in memory with name: rf_optuna_8fold_search_numeric_ready
Best trial: 0. Best value: 0.7665: 100%|██████████| 1/1 [26:56<00:00, 1616.12s/it]


[I 2026-01-26 05:44:39,926] Trial 0 finished with value: 0.7665002196710491 and parameters: {'n_estimators': 500, 'max_depth_choice': 'None', 'min_samples_split': 13, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'float', 'max_features_float': 0.8659541126403374, 'max_samples': 0.6061695553391381}. Best is trial 0 with value: 0.7665002196710491.
Melhor AUC (CV interna 8 folds): 0.7665002196710491
Melhores hiperparâmetros: {'n_estimators': 500, 'min_samples_split': 13, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_samples': 0.6061695553391381, 'max_depth': None, 'max_features': 0.8659541126403374, 'n_jobs': 8, 'random_state': 42}

Melhor Threshold para F1 (na validação): 0.3684

 RESULTADO FINAL NA BASE DE TESTE (Fold 10) 
 - ACC: 0.6842
 - F1: 0.6180
 - BAC: 0.6956
 - PRE: 0.5341
 - REC: 0.7331
 - AUC: 0.7636
 - CE-C0: 0.3811
 - CE-C1: 0.8528
 - TP%: 25.5400
 - TN%: 42.8844
 - FP%: 22.2787
 - FN%: 9.2969
 - TP_abs: 2412
 - T

In [ ]:
#{'n_estimators': 900, 'min_samples_split': 4, 'min_samples_leaf': 3, 'bootstrap': False, 'class_weight': 'balanced', 'max_depth': None, 'max_features': 'log2', 'n_jobs': -1, 'random_state': 42}
rf_best_params

In [ ]:
# ====================== FASE FINAL 9/1 (usa rf_best_params e split 8/1/1 já definido) ======================
# Requisitos (já existentes no ambiente a partir do código 8/1/1):
# - df, EXCLUDE_COLUMNS, THRESHOLD, CSV_OUT (Path), MODEL_DIR (Path), DATASET_NAME (str)
# - rf_best_params  (melhores hiperparâmetros escolhidos pelo Optuna)
# - X_train9, y_train9  (TREINO = 8 folds + validação)
# - X_test1,  y_test1   (TESTE final, fold 10 nunca visto)
# - FEATURES, id_cols (ou ID_COLS compatível)

import numpy as np, pandas as pd, time, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score, accuracy_score,
    f1_score, precision_score, recall_score, balanced_accuracy_score, log_loss
)

# Se N_SPLITS não existir, usa o mesmo do outer (10)
try:
    N_SPLITS
except NameError:
    N_SPLITS = N_SPLITS_OUTER if 'N_SPLITS_OUTER' in globals() else 10

# -------------------- Preparação dos dados X, y --------------------
y = df['y']

# Reusa a lógica de exclusão já utilizada
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != 'y']

# FEATURES já deve existir, mas garantimos aqui
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

# -------------------- Reaproveita o split 9/1 já definido --------------------
# TREINO = índices de X_train9 (8 folds + validação)
# TESTE  = índices de X_test1  (fold 10 nunca visto)
train_idx = X_train9.index.to_numpy()
test_idx  = X_test1.index.to_numpy()

# -------------------- Monta conjuntos mantendo IDs para rastreio --------------------
if len(ID_COLS) > 0:
    X_train_full = df.loc[train_idx, ID_COLS + FEATURES].copy()
    X_test_full  = df.loc[test_idx,  ID_COLS + FEATURES].copy()
else:
    X_train_full = df.loc[train_idx, FEATURES].copy()
    X_test_full  = df.loc[test_idx,  FEATURES].copy()

X_train = X_train_full[FEATURES].copy()
X_test  = X_test_full[FEATURES].copy()

# Reusa y diretamente a partir dos índices
y_train = y.loc[train_idx].copy()
y_test  = y.loc[test_idx].copy()

# -------------------- Treino do modelo final (9 folds = 8+1) --------------------
t0 = time.time()

# Reusa os melhores hiperparâmetros encontrados na fase 8/1/1
rf_params = rf_best_params

model = RandomForestClassifier(**rf_params)
model.fit(X_train, y_train)

# -------------------- Probabilidades e predições --------------------
proba_test_all  = model.predict_proba(X_test)
proba_test_c1   = proba_test_all[:, 1]
pred_test_bin   = (proba_test_c1 >= THRESHOLD).astype(int)

proba_train_all = model.predict_proba(X_train)
proba_train_c1  = proba_train_all[:, 1]
pred_train_bin  = (proba_train_c1 >= THRESHOLD).astype(int)

# -------------------- Armazena blocos (compatível com seu pipeline) --------------------
X_test_parts  = [X_test_full.assign(fold=10).reset_index()]
y_test_parts  = [pd.Series(y_test,  name='y_test').to_frame().assign(fold=10).reset_index()]
y_test_pred_parts  = [pd.DataFrame(proba_test_all,  columns=['prob_0','prob_1'],
                                   index=X_test_full.index).assign(fold=10).reset_index()]

X_train_parts = [X_train_full.assign(fold=9).reset_index()]
y_train_parts = [pd.Series(y_train, name='y_train').to_frame().assign(fold=9).reset_index()]
y_train_pred_parts = [pd.DataFrame(proba_train_all, columns=['prob_0','prob_1'],
                                   index=X_train_full.index).assign(fold=9).reset_index()]

# -------------------- Métricas (com Cross-Entropy por classe e %) --------------------
resultados = []

# --- TREINO (9 folds = 8+1) ---
cm_tr = confusion_matrix(y_train, pred_train_bin)
tn_tr, fp_tr, fn_tr, tp_tr = cm_tr.ravel()
total_tr = cm_tr.sum()
auroc_tr = roc_auc_score(y_train, proba_train_c1)

mask0_tr = (y_train == 0); mask1_tr = ~mask0_tr
ll_tr_0 = (log_loss(y_train[mask0_tr], proba_train_c1[mask0_tr], labels=[0,1])
           if mask0_tr.sum() > 0 else np.nan)
ll_tr_1 = (log_loss(y_train[mask1_tr], proba_train_c1[mask1_tr], labels=[0,1])
           if mask1_tr.sum() > 0 else np.nan)
prop_tr = y_train.value_counts(normalize=True) * 100

resultados.append({
    'Conjunto': 'Treino(9folds)', 'Fold': 9,
    'Acurácia': accuracy_score(y_train, pred_train_bin),
    'Cross-Entropy C0': ll_tr_0,
    'Proporção C0': prop_tr.get(0, np.nan),
    'Cross-Entropy C1': ll_tr_1,
    'Proporção C1': prop_tr.get(1, np.nan),
    'F1 Score': f1_score(y_train, pred_train_bin),
    'Balanced Accuracy': balanced_accuracy_score(y_train, pred_train_bin),
    'Precision': precision_score(y_train, pred_train_bin),
    'Recall': recall_score(y_train, pred_train_bin),
    'AUROC': auroc_tr,
    'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
    'TP(%)': round(tp_tr/total_tr*100, 2), 'FP(%)': round(fp_tr/total_tr*100, 2),
    'TN(%)': round(tn_tr/total_tr*100, 2), 'FN(%)': round(fn_tr/total_tr*100, 2),
    'Tempo (s)': round(time.time() - t0, 2)
})

# --- TESTE (fold 10, nunca visto) ---
cm_ts = confusion_matrix(y_test, pred_test_bin)
tn_ts, fp_ts, fn_ts, tp_ts = cm_ts.ravel()
total_ts = cm_ts.sum()
auroc_ts = roc_auc_score(y_test, proba_test_c1)

mask0_ts = (y_test == 0); mask1_ts = ~mask0_ts
ll_ts_0 = (log_loss(y_test[mask0_ts], proba_test_c1[mask0_ts], labels=[0,1])
           if mask0_ts.sum() > 0 else np.nan)
ll_ts_1 = (log_loss(y_test[mask1_ts], proba_test_c1[mask1_ts], labels=[0,1])
           if mask1_ts.sum() > 0 else np.nan)
prop_ts = y_test.value_counts(normalize=True) * 100

resultados.append({
    'Conjunto': 'Teste(fold10)', 'Fold': 10,
    'Acurácia': accuracy_score(y_test, pred_test_bin),
    'Cross-Entropy C0': ll_ts_0,
    'Proporção C0': prop_ts.get(0, np.nan),
    'Cross-Entropy C1': ll_ts_1,
    'Proporção C1': prop_ts.get(1, np.nan),
    'F1 Score': f1_score(y_test, pred_test_bin),
    'Balanced Accuracy': balanced_accuracy_score(y_test, pred_test_bin),
    'Precision': precision_score(y_test, pred_test_bin),
    'Recall': recall_score(y_test, pred_test_bin),
    'AUROC': auroc_ts,
    'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
    'TP(%)': round(tp_ts/total_ts*100, 2), 'FP(%)': round(fp_ts/total_ts*100, 2),
    'TN(%)': round(tn_ts/total_ts*100, 2), 'FN(%)': round(fn_ts/total_ts*100, 2),
    'Tempo (s)': round(time.time() - t0, 2)
})

# -------------------- Importâncias de features --------------------
importancias_lista = []
importancias = model.feature_importances_
linha_imp = {'Conjunto': 'Teste(fold10)', 'Fold': 10}
for nome, valor in zip(FEATURES, importancias):
    linha_imp[nome] = f"{valor * 100:.2f}%"
importancias_lista.append(linha_imp)

# -------------------- Salva resultados principais --------------------
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}_9_1.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}_9_1.csv', index=False)

# -------------------- Gera datasets com probabilidades (compatível com seu fluxo) --------------------
X_train_global      = pd.concat(X_train_parts, ignore_index=True)
y_train_global      = pd.concat(y_train_parts, ignore_index=True)
y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True)
X_test_global       = pd.concat(X_test_parts, ignore_index=True)
y_test_global       = pd.concat(y_test_parts, ignore_index=True)
y_test_pred_global  = pd.concat(y_test_pred_parts, ignore_index=True)

for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

train_df = X_train_global.merge(y_train_global, on=['orig_index','fold'], how='left')
train_df = train_df.merge(y_train_pred_global, on=['orig_index','fold'], how='left')
test_df  = X_test_global.merge(y_test_global, on=['orig_index','fold'], how='left')
test_df  = test_df.merge(y_test_pred_global, on=['orig_index','fold'], how='left')

train_df['y_proba'] = train_df['prob_1']; train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
test_df['y_proba']  = test_df['prob_1'];  test_df['y_pred']  = (test_df['y_proba']  >= THRESHOLD).astype(int)

train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}_9_1.csv', index=False)
test_df.to_csv (CSV_OUT / f'rf_test_with_proba_{N_SPLITS}_9_1.csv',  index=False)

# (Opcional) dataset unificado com previsões no conjunto original
df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
preds  = test_df[['orig_index','y_pred','y_proba']]
df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_{N_SPLITS}_9_1.csv', index=False)

# -------------------- Salva modelo final --------------------
model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}_9_1.pkl'
joblib.dump(model, model_path)

print('✅ Fase final 9/1 concluída. CSVs e modelo salvos em:', CSV_OUT, 'e', model_path)
print(df_resultados)


In [ ]:
importances_df = pd.DataFrame({
    'Feature': model.feature_names_in_,
    'Importance': model.feature_importances_
})

# 2. Ordena o DataFrame pela importância em ordem decrescente
importances_df = importances_df.sort_values(by='Importance', ascending=False)

# 3. Exibe a tabela
print("Tabela de Importância das Features (ordenada):")
print(importances_df)



In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ========================= Geração de PDF resumo (9/1) =========================
with PdfPages(PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # -------------------- Página de parâmetros --------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = rf_params.copy()
    parametros['threshold']   = THRESHOLD
    parametros['n_splits']    = N_SPLITS
    # esquema 9/1
    parametros['schema']      = '9/1 (treino: 9 folds, teste: 1 fold)'
    parametros['train_folds'] = 9
    parametros['test_folds']  = 1

    texto = 'Algoritmo: Random Forest\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - Random Forest')

    # salva PNG dos parâmetros (opcional)
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig)
    plt.close()

    # -------------------- Página de métricas resumidas --------------------
    if not df_resultados.empty:
        # normaliza rótulos Treino/Teste para agrupar
        tmp = df_resultados.copy()
        tmp['Conjunto_norm'] = tmp['Conjunto'].astype(str)
        tmp.loc[tmp['Conjunto_norm'].str.startswith('Treino'), 'Conjunto_norm'] = 'Treino'
        tmp.loc[tmp['Conjunto_norm'].str.startswith('Teste'),  'Conjunto_norm'] = 'Teste'
        display_df = tmp.groupby(['Conjunto_norm']).mean(numeric_only=True).T.round(4)

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        table = ax.table(
            cellText=display_df.values,
            colLabels=display_df.columns,
            rowLabels=display_df.index,
            loc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig)
        plt.close()

    # -------------------- Página ROC - Treino vs Teste (lado a lado) --------------------
    try:
        plotted = False
        # Tenta usar dataframes concatenados quando disponíveis (train_df/test_df) – 9/1 já produz esses
        if 'y_train' in train_df.columns and 'prob_1' in train_df.columns:
            y_train_all = train_df['y_train']
            y_score_train_all = train_df['prob_1']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train)
            plotted = True
        elif 'fprs_train' in globals() and len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if len(aurocs_train) > 0 else auc(fpr_train, tpr_train)
            plotted = True

        if 'y_test' in test_df.columns and 'prob_1' in test_df.columns:
            y_test_all = test_df['y_test']
            y_score_test_all = test_df['prob_1']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test)
            plotted = True
        elif 'fprs_test' in globals() and len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if len(aurocs_test) > 0 else auc(fpr_test, tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # ROC Treino (esquerda)
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[0].set_title('ROC - Treino (9 folds)')
                axes[0].set_xlabel('FPR')
                axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5, 0.5, 'Não há dados de treino para ROC', ha='center')
            # ROC Teste (direita)
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[1].set_title('ROC - Teste (fold 10)')
                axes[1].set_xlabel('FPR')
                axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5, 0.5, 'Não há dados de teste para ROC', ha='center')
            plt.tight_layout()
            # salva PNG do ROC combinado
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # -------------------- Página Confusion Matrix - Treino vs Teste --------------------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train, cm_train_pct = None, None
        cm_test, cm_test_pct = None, None

        if 'y_train' in train_df.columns and 'y_pred' in train_df.columns:
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        else:
            # fallback: busca linhas cujo 'Conjunto' comece com 'Treino'
            if 'TP' in df_resultados.columns:
                train_rows = df_resultados[df_resultados['Conjunto'].astype(str).str.startswith('Treino')]
                TP = int(train_rows['TP'].sum()) if not train_rows.empty else 0
                FP = int(train_rows['FP'].sum()) if not train_rows.empty else 0
                TN = int(train_rows['TN'].sum()) if not train_rows.empty else 0
                FN = int(train_rows['FN'].sum()) if not train_rows.empty else 0
                cm_train = np.array([[TN, FP], [FN, TP]])
                total = cm_train.sum()
                cm_train_pct = cm_train / total * 100 if total > 0 else np.zeros_like(cm_train, dtype=float)

        if 'y_test' in test_df.columns and 'y_pred' in test_df.columns:
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        else:
            # fallback: busca linhas cujo 'Conjunto' comece com 'Teste'
            if 'TP' in df_resultados.columns:
                test_rows = df_resultados[df_resultados['Conjunto'].astype(str).str.startswith('Teste')]
                TP = int(test_rows['TP'].sum()) if not test_rows.empty else 0
                FP = int(test_rows['FP'].sum()) if not test_rows.empty else 0
                TN = int(test_rows['TN'].sum()) if not test_rows.empty else 0
                FN = int(test_rows['FN'].sum()) if not test_rows.empty else 0
                cm_test = np.array([[TN, FP], [FN, TP]])
                total = cm_test.sum()
                cm_test_pct = cm_test / total * 100 if total > 0 else np.zeros_like(cm_test, dtype=float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center')
                    ax.axis('off')
                    return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito')
                ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1'])
                ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (9 folds)')
            plot_cm(axes[1], cm_test, cm_test_pct,   'Matriz de Confusão - Teste (fold 10)')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # -------------------- Importâncias (manteve igual; sem mudanças lógicas) --------------------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%','', regex=True)
        cols = [c for c in df_imp.columns if c not in ['Conjunto','Fold']]
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]
        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)
            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)')
            ax.set_title('Importância média das features (porcentagem)')
            plt.tight_layout()
            plt.subplots_adjust(left=0.30)
            try:
                png_path = IMAGES_OUT / f'feature_importances_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()

            # Tabela das importâncias
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                table_height = max(4, 0.3 * len(table_df))
                fig, ax = plt.subplots(figsize=(8, table_height))
                ax.axis('off')
                table = ax.table(
                    cellText=table_df.values,
                    colLabels=table_df.columns,
                    rowLabels=table_df.index,
                    loc='center'
                )
                table.auto_set_font_size(False)
                table.set_fontsize(9)
                table.scale(1, 1.2)
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout()
                pdf.savefig(fig, bbox_inches='tight')
                plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # (resto do Pareto / variância / violinos você pode manter igual,
            # não depende do ajuste 9/1)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# === Random Forest Basico (single train/test split) — AJUSTE 9/1 ===
try:
    RT_RANDOM_STATE
except NameError:
    # usa o mesmo RANDOM_STATE global do restante do pipeline, se existir
    RT_RANDOM_STATE = RANDOM_STATE if 'RANDOM_STATE' in globals() else 42

# Mantemos TEST_SIZE apenas para nome de arquivos (a divisão real é 9/1 via KFold)
TEST_SIZE = 0.1

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import pandas as pd
import joblib
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# Prepara X e y — usa FEATURES já detectadas no notebook e df
if 'FEATURES' not in globals():
    raise RuntimeError('FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('Dataframe df não encontrado — execute as células anteriores para carregar os dados')

X = df[FEATURES].copy()
y = df['y'].copy()

# ==== Split 9/1 determinístico (substitui o antigo train_test_split) ====
N_SPLITS_BASIC = 10  # 9/1
kf_basic = StratifiedKFold(n_splits=N_SPLITS_BASIC, shuffle=True, random_state=RT_RANDOM_STATE)
fold_pairs = list(kf_basic.split(X, y))

# último fold => teste; todos os demais => treino
_, test_idx = fold_pairs[-1]
all_idx = np.arange(len(X))
train_idx = np.setdiff1d(all_idx, test_idx)

# preserva índices originais (equivalente ao que você fazia com reset_index)
orig_train_idx = X.iloc[train_idx].index.values
orig_test_idx  = X.iloc[test_idx].index.values

X_train = X.iloc[train_idx].copy().set_index(pd.Index(orig_train_idx))
X_test  = X.iloc[test_idx].copy().set_index(pd.Index(orig_test_idx))
y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

# Treina modelo — usa os mesmos hiperparâmetros finais (rf_params)
model_basic = RandomForestClassifier(**rf_params)
model_basic.fit(X_train, y_train)

# Predições binárias
y_pred_train_bin = model_basic.predict(X_train)
y_pred_test_bin  = model_basic.predict(X_test)

# Probabilidades (para AUROC)
y_proba_train = model_basic.predict_proba(X_train)[:, 1]
y_proba_test  = model_basic.predict_proba(X_test)[:, 1]

# Acurácias e outras métricas
acc_train = accuracy_score(y_train, y_pred_train_bin)
acc_test  = accuracy_score(y_test, y_pred_test_bin)
f1_tr     = f1_score(y_train, y_pred_train_bin)
f1_ts     = f1_score(y_test, y_pred_test_bin)
prec_tr   = precision_score(y_train, y_pred_train_bin)
prec_ts   = precision_score(y_test, y_pred_test_bin)
rec_tr    = recall_score(y_train, y_pred_train_bin)
rec_ts    = recall_score(y_test, y_pred_test_bin)

auc_tr = roc_auc_score(y_train, y_proba_train) if len(np.unique(y_train)) > 1 else np.nan
auc_ts = roc_auc_score(y_test, y_proba_test)   if len(np.unique(y_test)) > 1 else np.nan

print(f"[9/1] Acurácia treino: {acc_train:.4f} — teste: {acc_test:.4f}")

# Salva modelo .pkl e dados
model_name = MODEL_DIR / f'rf_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl'
joblib.dump(model_basic, model_name)

# salva CSVs (mantém seus caminhos/nomes)
X_train.to_csv(BASICO_CSV / f'X_train_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
X_test.to_csv (BASICO_CSV / f'X_test_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
pd.Series(y_test,  name='y_test').to_csv (BASICO_CSV / f'y_test_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv',  index=True)
pd.Series(y_train, name='y_train').to_csv(BASICO_CSV / f'y_train_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=True)

# Salva features
features = X.columns.tolist()
joblib.dump(features, MODEL_DIR / f'features_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl')

# Salva métricas em CSV
metrics_basic = pd.DataFrame([{ 
    'model': str(model_name.name),
    'schema': '9/1 via KFold (último fold = teste)',
    'test_size_alias': TEST_SIZE,  # só informativo nos nomes
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train, 'acc_test': acc_test,
    'f1_train': f1_tr,     'f1_test': f1_ts,
    'precision_train': prec_tr, 'precision_test': prec_ts,
    'recall_train': rec_tr,     'recall_test': rec_ts,
    'auc_train': auc_tr,        'auc_test': auc_ts
}])
metrics_basic.to_csv(BASICO_CSV / f'rf_model_basic.csv', index=False)

# === Adiciona metrics_basic ao final do PDF resumo (mantido) ===
metrics_pdf_path = BASICO_DIR / f'rf_basic_metrics_page_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pdf'
main_pdf = PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}.pdf'
merged_pdf = PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}_merged.pdf'
try:
    fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
    ax.axis('off')
    display_df = metrics_basic.copy().round(4)
    table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    ax.set_title('Tabela: Métricas - Random Forest Básico (9/1)', pad=20)
    plt.tight_layout()
    with PdfPages(metrics_pdf_path) as pdf2:
        pdf2.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    try:
        from PyPDF2 import PdfMerger
        merger = PdfMerger()
        if main_pdf.exists():
            merger.append(str(main_pdf))
        merger.append(str(metrics_pdf_path))
        with open(merged_pdf, 'wb') as f:
            merger.write(f)
        merged_pdf.replace(main_pdf)
        if metrics_pdf_path.exists():
            metrics_pdf_path.unlink()
        print('metrics_basic (9/1) adicionada ao final do PDF resumo.')
    except Exception as e_merge:
        print('Não foi possível mesclar PDFs automaticamente. PDF de métricas salvo em:', metrics_pdf_path, '| Erro:', e_merge)
except Exception as e:
    print('Não foi possível gerar a página PDF de metrics_basic:', e)

# Dataset augmentado (original + previsões de TESTE)
try:
    X_test_reset = X_test.reset_index().rename(columns={'index': 'orig_index'})
    preds_df = pd.DataFrame({'orig_index': X_test_reset['orig_index'], 'y_pred': y_pred_test_bin, 'y_proba': y_proba_test})
    df_orig = df.reset_index().rename(columns={'index': 'orig_index'})
    df_augmented = pd.merge(df_orig, preds_df, how='left', on='orig_index')
    df_augmented.to_csv(BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
    df_augmented.to_csv(CSV_OUT    / f'database_used_{DATASET_NAME}_with_preds_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
except Exception as e_aug:
    print('Não foi possível gerar dataset augmentado do basico (9/1):', e_aug)

# Seleciona exemplos aleatórios do X_test para explicação (12 por padrão)
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_test.index, size=min(num_instancias, len(X_test)), replace=False)

print('Modelo salvo em:', model_name)
print('Métricas salvas em:', BASICO_CSV / f'rf_model_basic.csv')
print('Dataset augmentado salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
print('Índices amostra aleatorios uso diverso:', indices_aleatorios)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# 1) Correlação só das features (removendo id, alvo etc.)
df_feat = df.drop(columns=EXCLUDE_COLUMNS, errors="ignore")
df_corr = df_feat.corr(method="pearson")

# 2) Máscara para mostrar apenas o triângulo inferior
mask = np.triu(np.ones_like(df_corr, dtype=bool))

# 3) Estilo geral
sns.set(style="white")

# Aumenta bem a figura pra deixar os quadrados grandes
n_vars = df_corr.shape[0]
fig_size = max(10, n_vars * 0.6)  # escala simples pelo nº de variáveis

plt.figure(figsize=(fig_size, fig_size))

ax = sns.heatmap(
    df_corr,
    mask=mask,
    cmap="coolwarm",
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6},     # fonte dos números bem menor
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Correlação de Pearson"}
)

# 4) Ajusta fonte dos labels dos eixos (menor e mais limpa)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

plt.title("Mapa de calor – Correlações de Pearson (triângulo inferior)", fontsize=12, pad=16)

# 5) Destaca correlações fortes (|r| ≥ 0.6, fora da diagonal) com contorno
threshold = 0.6
for i in range(df_corr.shape[0]):
    for j in range(df_corr.shape[1]):
        if mask[i, j]:
            continue  # pula triângulo superior
        if i == j:
            continue  # ignora diagonal
        valor = df_corr.iloc[i, j]
        if abs(valor) >= threshold:
            ax.add_patch(
                Rectangle(
                    (j, i),     # x, y
                    1, 1,
                    fill=False,
                    edgecolor="black",
                    linewidth=1.8
                )
            )

plt.tight_layout()
plt.show()


In [ ]:
# Salvar X_test_basic_full.csv e X_train_basic_full.csv com todas as colunas + y, y_pred, y_proba (esquema 9/1)
import pandas as pd

# Carrega o arquivo cru (mesma ordem do df original)
raw_df = pd.read_csv(DATASET_PATH)

# Garante que existe coluna 'y' no raw_df, de forma genérica
if 'y' not in raw_df.columns:
    if 'septic' in raw_df.columns:  # caso específico WDBC
        raw_df['y'] = raw_df['septic'].map({'1': 1, '0': 0})
    else:
        raise RuntimeError(
            "Coluna 'y' não encontrada em raw_df e não há coluna 'diagnosis' para mapear. "
            "Ajuste a criação de 'y' para este dataset."
        )

# Usa o modelo final do 9/1 (clf_final) com X_test1 / X_train9
# (clf_final é o Pipeline treinado em X_train9 / y_train9 no bloco 9/1)
proba_test  = clf_final.predict_proba(X_test1)[:, 1]
y_pred_test = (proba_test >= THRESHOLD).astype(int)

proba_train  = clf_final.predict_proba(X_train9)[:, 1]
y_pred_train = (proba_train >= THRESHOLD).astype(int)

# Função para montar e salvar o DataFrame completo, alinhando por índice
def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    df_full = raw_df.loc[indices].copy()

    # Alinha vetores ao índice original
    y_true_s  = pd.Series(y_true,  index=indices)
    y_pred_s  = pd.Series(y_pred,  index=indices)
    y_proba_s = pd.Series(y_proba, index=indices)

    df_full['y']       = y_true_s
    df_full['y_pred']  = y_pred_s
    df_full['y_proba'] = y_proba_s

    df_full.to_csv(BASICO_CSV / f'X_{split_name}_basic_full.csv', index=True)

# Salva usando os índices de treino/teste do esquema 9/1
save_full_split(X_test1.index,  y_test1,  y_pred_test,  proba_test,  'test')
save_full_split(X_train9.index, y_train9, y_pred_train, proba_train, 'train')
